In [28]:
import sys
!{sys.executable} -m pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable


In [29]:
import numpy as np
import pandas as pd
import os
import Dataset_Download
import cleaning
import recommender_system
import SentimentScore


In [2]:
Dataset_Download.download()


All Kaggle datasets processed!


In [3]:
folder_path=os.path.join("Datasets", "the-movies-dataset")
files=['credits','keywords','links','links_small','movies_metadata','ratings_small','ratings']
dataframes={}
for file in files:
    file_path=os.path.join(folder_path, f"{file}.csv")
    dataframes[file] = pd.read_csv(file_path,low_memory=False)

In [4]:
# i have not downloaded the movielens_25m dataset here

In [5]:
credits_df=dataframes['credits']
keywords_df=dataframes['keywords']
links_df=dataframes['links']
links_small_df=dataframes['links_small']
movies_metadata_df=dataframes['movies_metadata']
ratings_df=dataframes['ratings']
ratings_small_df=dataframes['ratings_small']

In [6]:
combined_links = pd.concat([links_df, links_small_df], axis=0, ignore_index=True)

In [7]:
combined_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
combined_ratings = pd.concat([ratings_df, ratings_small_df], axis=0, ignore_index=True)

In [9]:
combined_ratings.shape

(26124293, 4)

In [10]:
#to convert id column to numeric column
movies_metadata_df["id"] = pd.to_numeric(movies_metadata_df["id"], errors="coerce")

In [11]:
df = credits_df.merge(keywords_df,on="id", how="outer") \
        .merge(combined_links, left_on="id", right_on="tmdbId", how="outer") \
        .merge(movies_metadata_df,on="id", how="outer")

# Data Cleaning

In [12]:
df1=df.copy()

In [13]:
cleaner=cleaning.DataCleaner()

In [14]:
df1=cleaner.clean(df1)

c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\cleaning.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['imdbId'] = df['imdbId'].astype('Int64')
c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\cleaning.py:227: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['imdb_id'], inplace=True)
c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\cleaning.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [15]:
combined_ratings

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556
...,...,...,...,...
26124288,671,6268,2.5,1065579370
26124289,671,6269,4.0,1065149201
26124290,671,6365,4.0,1070940363
26124291,671,6385,2.5,1070979663


In [16]:
#here rating become different dataframe after applying rating_transform
ratings=cleaner.rating_transform(combined_ratings)

In [17]:
ratings

,movieId,rating,counts,ave_rating
0,1,257606.0,66255,3.888099
1,2,84719.0,26167,3.237628
2,3,49398.0,15556,3.175495
3,4,8603.5,2994,2.873580
4,5,47171.0,15314,3.080253
...,...,...,...,...
45118,176267,4.0,1,4.000000
45119,176269,3.5,1,3.500000
45120,176271,5.0,1,5.000000
45121,176273,1.0,1,1.000000


# Recommendation

In [18]:
recommend=recommender_system.recommender_system(df1,combined_ratings)

In [19]:
print(recommend.recommend_tfidf("Avatar"))

c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\content_based_filtering.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[item]=new_df[item].apply(lambda x:' '.join(x))
c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\content_based_filtering.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[item]=new_df[item].apply(self.clean_text)


                           title     score
0                         Avatar  0.570000
1                       Avatar 2  0.510000
2            Yu-Gi-Oh! The Movie  0.406675
3                           Show  0.353992
4  Left Behind III: World at War  0.349577
5                   Maximum Ride  0.346638
6     Quest of the Delta Knights  0.340711
7                          Spawn  0.339753
8                      Darlings!  0.338256
9      Jimmy Neutron: Boy Genius  0.338018


In [20]:
# as 'harry potter' is not in the movie list
print(recommend.recommend_tfidf("harry potter"))

c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\content_based_filtering.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[item]=new_df[item].apply(lambda x:' '.join(x))
c:\Users\Khusi Patra\Desktop\Projects\Coding club\CINEIQ\content_based_filtering.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[item]=new_df[item].apply(self.clean_text)


Movie not found in dataset.
Empty DataFrame
Columns: []
Index: []


In [21]:
recommend.recommend_svd(1)

,movie,score
0,The Godfather,5.359592
1,The Godfather: Part II,3.776020
2,Fight Club,3.225135
3,The Dark Knight,2.850084
4,Inception,2.066465
5,The Sixth Sense,1.942461
6,Memento,1.726485
7,Batman Begins,1.461664
8,Iron Man,1.422339
9,The Dark Knight Rises,1.301108
